In [13]:
!pip install openpyxl

In [1]:
import glob
import re
import requests
import pandas as pd
import numpy as np

from bs4 import BeautifulSoup  # Vajalik peidetud väljade lugemiseks (pip install beautifulsoup4)
import io

In [6]:
def puhasta_exceli_kokku_read(faili_tee):
    print(f"Töötlen faili: {faili_tee}")
    
    # Loeme Exceli faili sisse ilma päisteta ja stringina, et vältida andmete nihkumist
    df = pd.read_excel(faili_tee, header=None, dtype=str)
    
    records = []
    current_time_block = None
    current_maakond = None
    
    # Käime läbi kõik Exceli read
    for index, row in df.iterrows():
        # Võtame väärtused veergude positsioonide järgi
        val1 = str(row[0]).strip() if pd.notna(row[0]) else ''
        val2 = str(row[1]).strip() if pd.notna(row[1]) else ''
        pindala = str(row[2]).strip() if pd.notna(row[2]) else ''
        arv = str(row[3]).strip() if pd.notna(row[3]) else ''
        keskmine = str(row[4]).strip() if pd.notna(row[4]) else ''
        
        # 1. Tuvastame aasta ja kvartali ploki (nt "2016 I", "2020 II")
        if re.match(r'^\d{4}\s+[IVX]+$', val1):
            current_time_block = val1
            
        # 2. Tuvastame maakonna nime
        if val2 and "KOKKU" not in val2.upper():
            current_maakond = val2
            
            
        # 3. Jätame alles AINULT need read, kus pindala veerus on "KOKKU"
        if pindala.upper() == "KOKKU":
            records.append({
                "Aeg_Plokk": current_time_block,
                "Maakond": current_maakond,
                "Tehingute_arv": arv,
                "Keskmine_hind": keskmine
            })
            
    if not records:
        return pd.DataFrame()
        
    df_tulemus = pd.DataFrame(records)
    
    # Puhastame arvud (asendame saladuslikud "***" väärtused tühja lahtriga / NULL-iga)
    df_tulemus["Tehingute_arv"] = pd.to_numeric(df_tulemus["Tehingute_arv"].str.replace(r'\s+', '', regex=True), errors='coerce')
    df_tulemus["Keskmine_hind"] = pd.to_numeric(df_tulemus["Keskmine_hind"].str.replace(r'\s+', '', regex=True), errors='coerce')
    
    return df_tulemus[["Aeg_Plokk", "Maakond", "Tehingute_arv", "Keskmine_hind"]]

# PANEME KÕIK EXCELI FAILID KOKKU ÜHTE CSV-SSE
exceli_failid = glob.glob("Kinnisvara hinnastatistika_*.xlsx")
koond_kokku_andmed = pd.DataFrame()

for fail in exceli_failid:
    try:
        puhastatud_df = puhasta_exceli_kokku_read(fail)
        koond_kokku_andmed = pd.concat([koond_kokku_andmed, puhastatud_df], ignore_index=True)
    except Exception as e:
        print(f"Viga faili {fail} töötlemisel: {e}")

# Kustutame võimalikud kattuvad duplikaatread ja sorteerime aja järgi
if not koond_kokku_andmed.empty:
    koond_kokku_andmed = koond_kokku_andmed.drop_duplicates()
    koond_kokku_andmed = koond_kokku_andmed.sort_values(by=["Aeg_Plokk", "Maakond"]).reset_index(drop=True)

# Salvestame lõpptulemuse puhta CSV-failina
koond_kokku_andmed.to_csv("maa_amet_py_final.csv", index=False, encoding='utf-8')
print(f"\nEdukas! Alles jäeti {len(koond_kokku_andmed)} koondrida ja salvestati faili 'maakondade_kokku_andmed.csv'.")

Töötlen faili: Kinnisvara hinnastatistika_05_08_raw.xlsx
Töötlen faili: Kinnisvara hinnastatistika_09_12_raw.xlsx
Töötlen faili: Kinnisvara hinnastatistika_13_15_raw.xlsx
Töötlen faili: Kinnisvara hinnastatistika_16_20_raw.xlsx
Töötlen faili: Kinnisvara hinnastatistika_21_25_raw.xlsx
Töötlen faili: Kinnisvara hinnastatistika_26_raw.xlsx

Edukas! Alles jäeti 1289 koondrida ja salvestati faili 'maakondade_kokku_andmed.csv'.
